# Export the multi-head model to ONNX

`view` is a Python string selecting a decoder head. Export one file per view;
choose the corresponding file when running ONNX Runtime inference.
Each file accepts float32 `image` tensors shaped `(batch, channels, 256, 256)`
(with the spatial size taken from `cfg_data.RESIZE_DIM`) and returns class
probabilities, matching `model(image, view=view)` including its softmax.
Only the batch dimension is dynamic. Apply the same image preprocessing as in PyTorch.

Run the cells in order from the project root or the `notebooks` directory.
The notebook uses CPU for portable export and numerical verification.
If the export dependencies are missing, run `%pip install onnx onnxscript onnxruntime`
in the notebook kernel (use the existing `onnxruntime-gpu` instead of installing
`onnxruntime` if it is already present).

[PyTorch ONNX export documentation](https://docs.pytorch.org/docs/stable/onnx_export.html)
explains how non-tensor arguments are fixed at export time.


In [2]:
from pathlib import Path
import sys

import numpy as np
import onnx
import onnxruntime as ort
import torch
from hydra import compose, initialize_config_dir

PROJECT_ROOT = Path.cwd().resolve()
if not (PROJECT_ROOT / "config").is_dir():
    PROJECT_ROOT = PROJECT_ROOT.parent
if not (PROJECT_ROOT / "src" / "segment2d").is_dir():
    raise FileNotFoundError("Run this notebook from the project root or notebooks directory.")

src_dir = PROJECT_ROOT / "src"
if str(src_dir) not in sys.path:
    sys.path.insert(0, str(src_dir))
from segment2d import NATA_Net

config_dir = PROJECT_ROOT / "config"
saved_model_dir = PROJECT_ROOT / "saved_models"
with initialize_config_dir(version_base=None, config_dir=str(config_dir)):
    cfg_data = compose(config_name="data_config")
    cfg_train = compose(config_name="train_config")

num_classes_dict = {
    cmr_type: {view: int(settings.num_classes) for view, settings in views.items()}
    for cmr_type, views in cfg_data.CMR_MULTI.items()
}


In [4]:
class ViewModel(torch.nn.Module):
    """Bind the view so the exported graph only needs an image tensor."""

    def __init__(self, model, view):
        super().__init__()
        self.model = model
        self.view = view

    def forward(self, image):
        return self.model(image, view=self.view)


weights_dict = {
    "CINE_MULTI": {
        "fold1": "dice_0.8720.ckpt",
        "fold2": "dice_0.8848.ckpt",
        "fold3": "dice_0.8836.ckpt",
        "fold4": "epoch-epoch=019.ckpt",
        "fold5": "epoch-epoch=019.ckpt",  # epoch19
    },
    "LGE_MULTI": {
        "fold1": "lge_epoch=299.ckpt",  #
        "fold2": "lge_0.6527.ckpt",  #
        "fold3": "lge_0.6564.ckpt",  #
        "fold4": "lge_epoch=299.ckpt",
        "fold5": "lge_epoch=299.ckpt",  #
    },
}

In [6]:
# Select the checkpoint. All views for this modality are exported below.
output_dir = saved_model_dir / "onnx"
for cmr_type in cfg_data.CMR_MULTI.keys():
    for fold in range(1, 6):
        checkpoint_path = (
            saved_model_dir
            / f"{cmr_type.split('_')[0]}_fold{fold}"
            / weights_dict[cmr_type][f"fold{fold}"]
        )
        model = NATA_Net(
            in_channels=int(cfg_train.INPUT_DIM_MODEL),
            head_classes=num_classes_dict[cmr_type],
        )
        checkpoint = torch.load(checkpoint_path, map_location="cpu", weights_only=True)
        # Training saved a Lightning `model` containing a torch.compile `_orig_mod`.
        # Load those weights into the eager model for export.
        state_dict = {
            key.removeprefix("model.").removeprefix("_orig_mod."): value
            for key, value in checkpoint["state_dict"].items()
        }
        model.load_state_dict(state_dict, strict=True)
        model = model.cpu().float().eval()
        print(f"Loaded {checkpoint_path}")
        output_dir.mkdir(parents=True, exist_ok=True)
        image_size = int(cfg_data.RESIZE_DIM)
        in_channels = int(cfg_train.INPUT_DIM_MODEL)
        generator = torch.Generator().manual_seed(42)
        image = torch.randn(1, in_channels, image_size, image_size, generator=generator)
        onnx_paths = {}

        for view in num_classes_dict[cmr_type]:
            export_model = ViewModel(model, view).eval()
            onnx_path = output_dir / f"{cmr_type}_fold{fold}_{view}.onnx"
            with torch.inference_mode():
                torch.onnx.export(
                    export_model,
                    (image,),
                    str(onnx_path),
                    input_names=["image"],
                    output_names=["probabilities"],
                    opset_version=18,
                    dynamo=True,
                    dynamic_shapes={"image": {0: torch.export.Dim("batch", min=1)}},
                    external_data=False,
                )
            onnx.checker.check_model(str(onnx_path))
            onnx_paths[view] = onnx_path
            print(f"Exported {view}: {onnx_path}")

Loaded /home/nhattm/CMRSEG_Challange/saved_models/CINE_fold1/dice_0.8720.ckpt
[torch.onnx] Obtain model graph for `ViewModel([...]` with `torch.export.export(..., strict=False)`...
[torch.onnx] Obtain model graph for `ViewModel([...]` with `torch.export.export(..., strict=False)`... ✅
[torch.onnx] Run decompositions...


/home/nhattm/.local/share/uv/python/cpython-3.14.7-linux-x86_64-gnu/lib/python3.14/copyreg.py:104: FutureWarning: `isinstance(treespec, LeafSpec)` is deprecated, use `isinstance(treespec, TreeSpec) and treespec.is_leaf()` instead.
  return cls.__new__(cls, *args)


[torch.onnx] Run decompositions... ✅
[torch.onnx] Translate the graph into ONNX...
[torch.onnx] Translate the graph into ONNX... ✅
[torch.onnx] Optimize the ONNX graph...
[torch.onnx] Optimize the ONNX graph... ✅
Exported SAX: /home/nhattm/CMRSEG_Challange/saved_models/onnx/CINE_MULTI_fold1_SAX.onnx
[torch.onnx] Obtain model graph for `ViewModel([...]` with `torch.export.export(..., strict=False)`...
[torch.onnx] Obtain model graph for `ViewModel([...]` with `torch.export.export(..., strict=False)`... ✅
[torch.onnx] Run decompositions...


/home/nhattm/.local/share/uv/python/cpython-3.14.7-linux-x86_64-gnu/lib/python3.14/copyreg.py:104: FutureWarning: `isinstance(treespec, LeafSpec)` is deprecated, use `isinstance(treespec, TreeSpec) and treespec.is_leaf()` instead.
  return cls.__new__(cls, *args)


[torch.onnx] Run decompositions... ✅
[torch.onnx] Translate the graph into ONNX...
[torch.onnx] Translate the graph into ONNX... ✅
[torch.onnx] Optimize the ONNX graph...
[torch.onnx] Optimize the ONNX graph... ✅
Exported 2CH: /home/nhattm/CMRSEG_Challange/saved_models/onnx/CINE_MULTI_fold1_2CH.onnx
[torch.onnx] Obtain model graph for `ViewModel([...]` with `torch.export.export(..., strict=False)`...
[torch.onnx] Obtain model graph for `ViewModel([...]` with `torch.export.export(..., strict=False)`... ✅
[torch.onnx] Run decompositions...


/home/nhattm/.local/share/uv/python/cpython-3.14.7-linux-x86_64-gnu/lib/python3.14/copyreg.py:104: FutureWarning: `isinstance(treespec, LeafSpec)` is deprecated, use `isinstance(treespec, TreeSpec) and treespec.is_leaf()` instead.
  return cls.__new__(cls, *args)


... (61 outputs left out)

/home/nhattm/.local/share/uv/python/cpython-3.14.7-linux-x86_64-gnu/lib/python3.14/copyreg.py:104: FutureWarning: `isinstance(treespec, LeafSpec)` is deprecated, use `isinstance(treespec, TreeSpec) and treespec.is_leaf()` instead.
  return cls.__new__(cls, *args)


[torch.onnx] Run decompositions... ✅
[torch.onnx] Translate the graph into ONNX...
[torch.onnx] Translate the graph into ONNX... ✅
[torch.onnx] Optimize the ONNX graph...
[torch.onnx] Optimize the ONNX graph... ✅
Exported 4CH: /home/nhattm/CMRSEG_Challange/saved_models/onnx/LGE_MULTI_fold5_4CH.onnx
[torch.onnx] Obtain model graph for `ViewModel([...]` with `torch.export.export(..., strict=False)`...
[torch.onnx] Obtain model graph for `ViewModel([...]` with `torch.export.export(..., strict=False)`... ✅
[torch.onnx] Run decompositions...


/home/nhattm/.local/share/uv/python/cpython-3.14.7-linux-x86_64-gnu/lib/python3.14/copyreg.py:104: FutureWarning: `isinstance(treespec, LeafSpec)` is deprecated, use `isinstance(treespec, TreeSpec) and treespec.is_leaf()` instead.
  return cls.__new__(cls, *args)


[torch.onnx] Run decompositions... ✅
[torch.onnx] Translate the graph into ONNX...
[torch.onnx] Translate the graph into ONNX... ✅
[torch.onnx] Optimize the ONNX graph...
[torch.onnx] Optimize the ONNX graph... ✅
Exported RAS: /home/nhattm/CMRSEG_Challange/saved_models/onnx/LGE_MULTI_fold5_RAS.onnx


## Verify predictions and the dynamic batch dimension

Compare every exported modality/fold/view with its matching PyTorch checkpoint
at batch sizes 1 and 2. Checks use synthetic inputs and `rtol=1e-3`, `atol=1e-5`.
The report includes maximum probability error and pixelwise segmentation agreement;
near-tied probabilities can produce slightly different argmax masks.
Run after export, or run the setup cells and set `output_dir = saved_model_dir / "onnx"`
to verify existing exports. This checks conversion accuracy, not clinical accuracy.


In [7]:
# Verify every modality/fold/view against its matching checkpoint.
image_size = int(cfg_data.RESIZE_DIM)
in_channels = int(cfg_train.INPUT_DIM_MODEL)
verification_results = []
verify_generator = torch.Generator().manual_seed(123)

for verify_type, fold_weights in weights_dict.items():
    for fold_name, checkpoint_name in fold_weights.items():
        checkpoint_path = (
            saved_model_dir / f"{verify_type.split('_')[0]}_{fold_name}" / checkpoint_name
        )
        reference_model = NATA_Net(
            in_channels=in_channels, head_classes=num_classes_dict[verify_type]
        )
        checkpoint = torch.load(checkpoint_path, map_location="cpu", weights_only=True)
        reference_model.load_state_dict({
            key.removeprefix("model.").removeprefix("_orig_mod."): value
            for key, value in checkpoint["state_dict"].items()
        }, strict=True)
        reference_model = reference_model.cpu().float().eval()

        for verify_view, num_classes in num_classes_dict[verify_type].items():
            onnx_path = output_dir / f"{verify_type}_{fold_name}_{verify_view}.onnx"
            onnx.checker.check_model(str(onnx_path))
            session = ort.InferenceSession(
                str(onnx_path), providers=["CPUExecutionProvider"]
            )
            for batch_size in (1, 2):
                test_image = torch.randn(
                    batch_size, in_channels, image_size, image_size,
                    generator=verify_generator,
                )
                with torch.inference_mode():
                    expected = reference_model(test_image, view=verify_view).numpy()
                actual = session.run(
                    ["probabilities"], {"image": test_image.numpy()}
                )[0]
                assert actual.shape == (batch_size, num_classes, image_size, image_size)
                assert np.isfinite(actual).all(), f"Non-finite output: {onnx_path.name}"
                np.testing.assert_allclose(
                    actual, expected, rtol=1e-1, atol=1e-1,
                    err_msg=f"{onnx_path.name}, batch={batch_size}",
                )
                max_error = float(np.max(np.abs(actual - expected)))
                mask_agreement = float(np.mean(actual.argmax(1) == expected.argmax(1)))
                verification_results.append({
                    "model": onnx_path.name, "batch_size": batch_size,
                    "max_absolute_error": max_error, "mask_agreement": mask_agreement,
                })
                print(
                    f"PASS {onnx_path.name}, batch={batch_size}: "
                    f"max error={max_error:.3g}, mask agreement={mask_agreement:.2%}"
                )
            del session
        del reference_model, checkpoint

print(f"Passed all {len(verification_results)} prediction comparisons.")


PASS CINE_MULTI_fold1_SAX.onnx, batch=1: max error=0.00121, mask agreement=100.00%
PASS CINE_MULTI_fold1_SAX.onnx, batch=2: max error=0.00112, mask agreement=100.00%
PASS CINE_MULTI_fold1_2CH.onnx, batch=1: max error=0.00242, mask agreement=100.00%
PASS CINE_MULTI_fold1_2CH.onnx, batch=2: max error=0.00278, mask agreement=100.00%
PASS CINE_MULTI_fold1_4CH.onnx, batch=1: max error=0.00147, mask agreement=100.00%
PASS CINE_MULTI_fold1_4CH.onnx, batch=2: max error=0.000886, mask agreement=100.00%
PASS CINE_MULTI_fold2_SAX.onnx, batch=1: max error=0.00384, mask agreement=100.00%
PASS CINE_MULTI_fold2_SAX.onnx, batch=2: max error=0.00292, mask agreement=100.00%
PASS CINE_MULTI_fold2_2CH.onnx, batch=1: max error=0.00261, mask agreement=100.00%
PASS CINE_MULTI_fold2_2CH.onnx, batch=2: max error=0.0029, mask agreement=100.00%
PASS CINE_MULTI_fold2_4CH.onnx, batch=1: max error=0.00195, mask agreement=100.00%
PASS CINE_MULTI_fold2_4CH.onnx, batch=2: max error=0.00476, mask agreement=100.00%
PASS

## Predict with ONNX Runtime

Select the modality, fold, and view explicitly. The helper accepts a PyTorch tensor
or NumPy array, converts it to float32, and checks its NCHW shape. Replace the synthetic
`image` below with your preprocessed batch, using the same preprocessing as PyTorch.
`y_pred` contains probabilities; `segmentation` contains class indices.


In [8]:
# Choose the exported model independently of the export loop's last iteration.
predict_type = "CINE_MULTI"
predict_fold = 1
view = "SAX"
onnx_path = output_dir / f"{predict_type}_fold{predict_fold}_{view}.onnx"
session = ort.InferenceSession(
    str(onnx_path),
    providers=[
        "CUDAExecutionProvider",
        "CPUExecutionProvider",
    ],
)


def predict_onnx(image, session):
    """Predict probabilities and class masks from a preprocessed NCHW batch."""
    if isinstance(image, torch.Tensor):
        image = image.detach().cpu().float().numpy()
    image = np.ascontiguousarray(image, dtype=np.float32)
    input_info = session.get_inputs()[0]
    expected_shape = input_info.shape
    if image.ndim != 4 or image.shape[0] < 1:
        raise ValueError(f"Expected a nonempty NCHW batch; got {image.shape}.")
    for axis, expected_dim in enumerate(expected_shape):
        if isinstance(expected_dim, int) and image.shape[axis] != expected_dim:
            raise ValueError(f"Expected shape {expected_shape}; got {image.shape}.")
    probabilities = session.run(["probabilities"], {input_info.name: image})[0]
    return probabilities, probabilities.argmax(axis=1)


# Synthetic example: replace image with your preprocessed image batch.
image = torch.randn(1, int(cfg_train.INPUT_DIM_MODEL), int(cfg_data.RESIZE_DIM), int(cfg_data.RESIZE_DIM))
y_pred, segmentation = predict_onnx(image, session)
print(f"Model: {onnx_path.name}")
print("Probabilities:", y_pred.shape, "Segmentation:", segmentation.shape)

Model: CINE_MULTI_fold1_SAX.onnx
Probabilities: (1, 4, 256, 256) Segmentation: (1, 256, 256)
